<a href="https://colab.research.google.com/github/DongAnYu/ai-tutor-rag-system/blob/main/notebooks/10-Adding_Reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [2]:
!pip install -q llama-index==0.14.0 openai==1.107.0 llama-index-finetuning==0.4.1 llama-index-postprocessor-cohere-rerank==0.5.1 \
                llama-index-embeddings-huggingface==0.6.1 llama-index-embeddings-cohere==0.6.1 cohere==5.18.0 \
                chromadb==1.0.21 llama-index-vector-stores-chroma==0.5.3 llama-index-llms-google-genai==0.5.0 jedi==0.19.2

In [6]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
# os.environ["CO_API_KEY"] = "YOUR_COHERE_API_KEY"
# cohere_key = os.environ["CO_API_KEY"]

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')
os.environ["CO_API_KEY"] = userdata.get('COHERE_API_KEY')
cohere_key = userdata.get('COHERE_API_KEY')

In [7]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

# Load a Model


In [8]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

### Download vector store

In [9]:
from huggingface_hub import hf_hub_download

vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

In [10]:
!unzip -o vectorstore.zip

Archive:  vectorstore.zip
   creating: ai_tutor_knowledge/
   creating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/length.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/index_metadata.pickle  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/link_lists.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/header.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/data_level0.bin  
  inflating: ai_tutor_knowledge/chroma.sqlite3  


In [11]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Create the vector index
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
index = VectorStoreIndex.from_vector_store(vector_store)

# Query Dataset


In [12]:

from llama_index.postprocessor.cohere_rerank import CohereRerank

# Define the Cohere Reranking object to return only the first two highest ranking chunks.

cohere_rerank3 = CohereRerank(top_n=2, api_key=cohere_key,model = 'rerank-english-v3.0')

In [13]:
# Define the ServiceCotext object to tie the LLM for generating final answer,
# and the embedding model to help with retrieving related nodes.
# The `node_postprocessors` function will be applied to the retrieved nodes.
query_engine = index.as_query_engine(
    similarity_top_k=5, node_postprocessors=[cohere_rerank3]
)

res = query_engine.query("Write about Retrieval Augmented Generation?")

In [14]:
res.response

'Retrieval-Augmented Generation (RAG) is a hybrid approach that combines information retrieval and large-scale generative models to produce more accurate, up-to-date, and contextually grounded responses. It addresses common weaknesses of standalone generative models—such as outdated knowledge and fact fabrication—by providing query-relevant external evidence that the generator can incorporate at inference time.\n\nCore components and workflow\n- Retrieval component\n  - Purpose: extract relevant information from external knowledge sources in response to a user query.\n  - Phases:\n    - Indexing: organize documents for efficient lookup. This can use inverted indexes for sparse retrieval or dense vector encodings for dense retrieval.\n    - Searching: fetch candidate documents or chunks based on the query using the chosen index. Rerankers are often applied to refine the ordering of retrieved items.\n  - Design considerations: how to chunk documents, what embedding types to use for seman

In [15]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 1c324686-fad7-4a41-bfd3-d44f9612ca91
Title	 Evaluation of Retrieval-Augmented Generation: A Survey:Research Paper Information
Text	 Authors: Hao Yu, Aoran Gan, Kai Zhang, Shiwei Tong, Qi Liu, and Zhaofeng Liu.Numerous studies of Retrieval-Augmented Generation (RAG) systems have emerged from various perspectives since the advent of Large Language Models (LLMs). The RAG system comprises two primary components: Retrieval and Generation. The retrieval component aims to extract relevant information from various external knowledge sources. It involves two main phases, indexing and searching. Indexing organizes documents to facilitate efficient retrieval, using either inverted indexes for sparse retrieval or dense vector encoding for dense retrieval. The searching component utilizes these indexes to fetch relevant documents based on the user's query, often incorporating the optional rerankers to refine the ranking of the retrieved documents. The generation component utilizes the retr

# Evaluate Cohere rerank

Evaluating reranking is about measuring how much a second-stage ranking model (e.g., Cohere Rerank) improves the quality of retrieved documents after your base retriever has done its initial recall. In a RAG pipeline, this step directly affects answer accuracy, grounding, and hallucination rate.


In a typical RAG flow:


1.   Retriever (vector / BM25 / hybrid) → returns top-K candidates based on embedding similarity.
2.   Reranker (cross-encoder, e.g., Cohere Rerank) → re-scores those candidates using the full query + full text.
3. LLM → generates the answer using only the highest-ranked chunks.

The reranker does not increase recall; it improves precision at top-K.




In [16]:
from huggingface_hub import hf_hub_download
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

# Download the evaluation dataset
hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="rag_eval_dataset_question_context_subset_50.json", repo_type="dataset", local_dir="./")
rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json("./rag_eval_dataset_question_context_subset_50.json")

(…)_dataset_question_context_subset_50.json: 0.00B [00:00, ?B/s]

In [17]:
import pandas as pd


#  A simple function to show the evaluation result.
def display_results_retriever(name, eval_results):
    """Display results from evaluate."""

    metric_dicts = []
    for eval_result in eval_results:
        metric_dict = eval_result.metric_vals_dict
        metric_dicts.append(metric_dict)

    full_df = pd.DataFrame(metric_dicts)

    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()

    metric_df = pd.DataFrame(
        {"Retriever Name": [name], "Hit Rate": [hit_rate], "MRR": [mrr]}
    )

    return metric_df

In [18]:
from llama_index.core.evaluation import RetrieverEvaluator

# We can evaluate the retievers with different top_k values.
for i in [2, 4, 6, 8, 10]:
    retriever = index.as_retriever(
        similarity_top_k=i, node_postprocessors=[cohere_rerank3]
    )
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    print(display_results_retriever(f"Retriever top_{i}", eval_results))

    Retriever Name  Hit Rate   MRR
0  Retriever top_2       0.6  0.57
    Retriever Name  Hit Rate       MRR
0  Retriever top_4      0.68  0.593333
    Retriever Name  Hit Rate       MRR
0  Retriever top_6       0.7  0.597333
    Retriever Name  Hit Rate       MRR
0  Retriever top_8      0.78  0.608405
     Retriever Name  Hit Rate       MRR
0  Retriever top_10       0.8  0.610627


Evaluating reranking answers this question: Given the same retrieved candidates, does reranking move the truly relevant chunks closer to the top?

Metrics we used:
1. Hit Rate @K: Measures whether at least one relevant document appears in the top-K results.

2. MRR (Mean Reciprocal Rank): Measures how early the first relevant document appears.